# MediGuide
## AI-Powered Analysis for Intelligent Healthcare Assistance

**Authors**
- Saurabh Kumbhar — 25204974
- Azim Hassan — 25203062

### Project Overview
MediGuide is a Retrieval-Augmented Generation (RAG) based healthcare assistance project.
It extracts medical information from PDF documents, splits the content into manageable
chunks, converts those chunks into embeddings, stores them in Pinecone, and uses an
OpenAI model through LangChain to answer questions from the retrieved context.

> **Important:** MediGuide is an academic/informational project. It must not be treated
> as a substitute for diagnosis, emergency care, or advice from a qualified healthcare professional.

## Pipeline

`Medical PDFs → Text Extraction → Cleaning → Chunking → Embeddings → Pinecone → Retrieval → OpenAI → Answer`

This notebook is organized so that every major stage of the RAG pipeline can be tested independently.

## 1. Environment and Project Setup

Before running this notebook:

1. Activate the `mediguide` Conda environment.
2. Install the packages from `requirements.txt`.
3. Create a `.env` file in the project root.
4. Add `OPENAI_API_KEY` and `PINECONE_API_KEY` to `.env`.
5. Place the medical PDF files inside the `data/` directory.

Example `.env` structure:

```text
OPENAI_API_KEY=your_openai_api_key
PINECONE_API_KEY=your_pinecone_api_key
```

Never commit the `.env` file to GitHub.

In [8]:
# Display the current working directory.
# This is useful for confirming where Jupyter started the notebook.
from pathlib import Path

print("Current working directory:", Path.cwd())

Current working directory: c:\Users\Shree\Desktop\MediGuide\projects-saurabh-azim


In [9]:
# If the notebook is inside a subfolder such as `research/` or `notebooks/`,
# this block attempts to locate the project root by looking for requirements.txt.

from pathlib import Path

current_path = Path.cwd()

if (current_path / "requirements.txt").exists():
    PROJECT_ROOT = current_path
elif (current_path.parent / "requirements.txt").exists():
    PROJECT_ROOT = current_path.parent
else:
    PROJECT_ROOT = current_path

DATA_DIR = PROJECT_ROOT / "data"

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)

Project root: c:\Users\Shree\Desktop\MediGuide\projects-saurabh-azim
Data directory: c:\Users\Shree\Desktop\MediGuide\projects-saurabh-azim\data


## 2. Import Required Libraries

In [10]:
# Standard library
import os
from typing import List

# Environment variables
from dotenv import load_dotenv

# LangChain document processing
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

# Pinecone
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

# OpenAI and RAG chain
from langchain_openai import ChatOpenAI
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

print("Imports loaded successfully.")

Imports loaded successfully.


## 3. Load Medical PDF Documents

`DirectoryLoader` scans the `data/` folder and `PyPDFLoader` extracts text from each PDF page.
Keeping document loading in its own function makes the pipeline easier to reuse later from the Flask application.

In [38]:
def load_pdf_files(data_directory: Path) -> List[Document]:
    """Load all PDF files from the supplied directory.

    Parameters
    ----------
    data_directory:
        Directory containing the medical PDF documents.

    Returns
    -------
    List[Document]
        LangChain Document objects extracted from the PDFs.
    """
    if not data_directory.exists():
        raise FileNotFoundError(
            f"Data directory not found: {data_directory}\n"
            "Create the directory and place your medical PDF files inside it."
        )

    loader = DirectoryLoader(
        str(data_directory),
        glob="*.pdf",
        loader_cls=PyPDFLoader,
        show_progress=True,
    )

    return loader.load()

In [12]:
# Load the medical knowledge-base documents.
extracted_data = load_pdf_files(DATA_DIR)

print(f"Pages/documents extracted: {len(extracted_data)}")

100%|██████████| 1/1 [00:40<00:00, 40.56s/it]

Pages/documents extracted: 637


In [13]:
# Preview one extracted document without printing the entire dataset.
if extracted_data:
    print("Source:", extracted_data[0].metadata.get("source"))
    print("\nText preview:\n")
    print(extracted_data[0].page_content[:1000])

Source: c:\Users\Shree\Desktop\MediGuide\projects-saurabh-azim\data\Medical_book.pdf

Text preview:




## 4. Keep Only Essential Metadata

PDF loaders may attach several metadata fields. For this project, only the source file is
required for basic traceability, so the documents are normalized before chunking.

In [14]:
def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """Return documents containing only their text and source metadata."""

    minimal_docs = []

    for doc in docs:
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": doc.metadata.get("source", "unknown")},
            )
        )

    return minimal_docs


minimal_docs = filter_to_minimal_docs(extracted_data)

print(f"Documents after metadata cleanup: {len(minimal_docs)}")

Documents after metadata cleanup: 637


## 5. Split Documents into Chunks

Large documents should not be sent to the model as one block. Chunking creates smaller,
overlapping sections that can be embedded and retrieved independently.

Current configuration:
- **Chunk size:** 500 characters
- **Chunk overlap:** 20 characters

In [15]:
def split_documents(
    documents: List[Document],
    chunk_size: int = 500,
    chunk_overlap: int = 20,
) -> List[Document]:
    """Split documents into smaller overlapping text chunks."""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )

    return text_splitter.split_documents(documents)


text_chunks = split_documents(minimal_docs)

print(f"Number of text chunks created: {len(text_chunks)}")

Number of text chunks created: 5859


In [16]:
# Preview the first chunk.
if text_chunks:
    print("Source:", text_chunks[0].metadata.get("source"))
    print("\nChunk preview:\n")
    print(text_chunks[0].page_content)

Source: c:\Users\Shree\Desktop\MediGuide\projects-saurabh-azim\data\Medical_book.pdf

Chunk preview:

The GALE
ENCYCLOPEDIA
of MEDICINE
SECOND EDITION


## 6. Create Hugging Face Embeddings

MediGuide uses `sentence-transformers/all-MiniLM-L6-v2`.

The model converts each text chunk into a **384-dimensional vector**.
Semantically similar text should have vectors that are close to one another.

In [17]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"


def create_embedding_model():
    """Create the Hugging Face embedding model used by MediGuide."""
    return HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)


embedding_model = create_embedding_model()

print("Embedding model loaded:", EMBEDDING_MODEL_NAME)

C:\Users\Shree\AppData\Local\Temp\ipykernel_4196\458427538.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)
c:\Users\Shree\anaconda3\envs\mediguide\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Shree\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISA

Embedding model loaded: sentence-transformers/all-MiniLM-L6-v2


In [18]:
# Quick embedding test.
sample_vector = embedding_model.embed_query("What is acne?")

print("Embedding vector dimension:", len(sample_vector))
print("First 5 values:", sample_vector[:5])

Embedding vector dimension: 384
First 5 values: [-0.10689964890480042, 0.03186367452144623, -0.050717614591121674, 0.09282449632883072, -0.01662909798324108]


## 7. Load API Keys Securely

The API keys are loaded from `.env`.

The notebook checks that the keys exist, but it deliberately does **not** print them.

In [19]:
# Load variables from PROJECT_ROOT/.env
load_dotenv(PROJECT_ROOT / ".env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

missing_keys = []

if not OPENAI_API_KEY:
    missing_keys.append("OPENAI_API_KEY")

if not PINECONE_API_KEY:
    missing_keys.append("PINECONE_API_KEY")

if missing_keys:
    raise ValueError(
        "Missing environment variable(s): "
        + ", ".join(missing_keys)
        + ". Add them to your .env file."
    )

print("Required API keys loaded successfully.")

Required API keys loaded successfully.


## 8. Connect to Pinecone

In [20]:
# Create the Pinecone client.
pc = Pinecone(api_key=PINECONE_API_KEY)

print("Pinecone client initialized.")

Pinecone client initialized.


## 9. Create or Connect to the MediGuide Vector Index

The embedding model produces 384-dimensional vectors, so the Pinecone index must use
the same dimension.

Cosine similarity is used to compare query and document vectors.

In [21]:
INDEX_NAME = "mediguide"
EMBEDDING_DIMENSION = 384

# Create the index only when it does not already exist.
if not pc.has_index(INDEX_NAME):
    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBEDDING_DIMENSION,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1",
        ),
    )
    print(f"Created Pinecone index: {INDEX_NAME}")
else:
    print(f"Using existing Pinecone index: {INDEX_NAME}")

pinecone_index = pc.Index(INDEX_NAME)

Created Pinecone index: mediguide


## 10. Add the Medical Knowledge Base to Pinecone

Run the indexing cell when:
- you create the index for the first time, or
- you intentionally want to upload/re-index your document chunks.

Repeatedly running it can insert the same material more than once depending on your indexing strategy.

In [ ]:
# Set this to True only when you want to upload the local PDF chunks.
UPLOAD_DOCUMENTS = False

if UPLOAD_DOCUMENTS:
    vector_store = PineconeVectorStore.from_documents(
        documents=text_chunks,
        embedding=embedding_model,
        index_name=INDEX_NAME,
    )
    print(f"Uploaded {len(text_chunks)} chunks to Pinecone.")
else:
    print("UPLOAD_DOCUMENTS is False — skipped document upload.")

Uploaded 5859 chunks to Pinecone.


## 11. Load the Existing Pinecone Vector Store

For normal chatbot usage, MediGuide connects to the existing index rather than uploading
the PDFs every time the notebook runs.

In [27]:
vector_store = PineconeVectorStore.from_existing_index(
    index_name=INDEX_NAME,
    embedding=embedding_model,
)

print("Connected to MediGuide vector store.")

Connected to MediGuide vector store.


## 12. Configure the Retriever

For every user question, the retriever returns the **3 most similar chunks** from the medical knowledge base.

In [28]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)

print("Retriever configured with k=3.")

Retriever configured with k=3.


In [29]:
# Test retrieval before involving the LLM.
test_query = "What is acne?"
retrieved_docs = retriever.invoke(test_query)

print(f"Retrieved {len(retrieved_docs)} documents for: {test_query}\n")

for number, doc in enumerate(retrieved_docs, start=1):
    print(f"--- Result {number} ---")
    print("Source:", doc.metadata.get("source", "unknown"))
    print(doc.page_content[:500])
    print()

Retrieved 3 documents for: What is acne?

--- Result 1 ---
Source: c:\Users\Shree\Desktop\MediGuide\projects-saurabh-azim\data\Medical_book.pdf
GALE ENCYCLOPEDIA OF MEDICINE 226
Acne
GEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26

--- Result 2 ---
Source: c:\Users\Shree\Desktop\MediGuide\projects-saurabh-azim\data\Medical_book.pdf
GALE ENCYCLOPEDIA OF MEDICINE 2 25
Acne
Acne vulgaris affecting a woman’s face. Acne is the general
name given to a skin disorder in which the sebaceous
glands become inflamed. (Photograph by Biophoto Associ-
ates, Photo Researchers, Inc. Reproduced by permission.)
GEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 25

--- Result 3 ---
Source: c:\Users\Shree\Desktop\MediGuide\projects-saurabh-azim\data\Medical_book.pdf
Acidosis see Respiratory acidosis; Renal
tubular acidosis; Metabolic acidosis
Acne
Definition
Acne is a common skin disease characterized by
pimples on the face, chest, and back. It occurs when the
pores of the skin become clogged with oil, dead

## 13. Configure the OpenAI Chat Model

In [30]:
# Temperature 0 keeps answers more focused and less variable for a
# document-grounded question-answering system.
chat_model = ChatOpenAI(
    model="gpt-4o",
    temperature=0,
    api_key=OPENAI_API_KEY,
)

print("OpenAI chat model initialized.")

OpenAI chat model initialized.


## 14. MediGuide System Prompt

The system prompt tells the model to:
- use the retrieved medical context,
- avoid inventing unsupported information,
- clearly state when the answer is unavailable,
- remain concise,
- avoid presenting the chatbot as a replacement for professional medical care.

In [31]:
system_prompt = """
You are MediGuide, an AI-powered healthcare information assistant.

Use only the supplied retrieved context to answer the user's question.
If the answer cannot be supported by the retrieved context, clearly say that
the available knowledge base does not contain enough information.

Give a clear, concise, and easy-to-understand response.
Do not invent medical facts, diagnoses, prescriptions, or treatment instructions.
For urgent symptoms or emergencies, advise the user to seek appropriate professional
or emergency medical care.

Retrieved context:
{context}
"""


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

print("MediGuide prompt configured.")

MediGuide prompt configured.


## 15. Build the Retrieval-Augmented Generation Chain

In [32]:
# The document chain sends retrieved context to the chat model.
question_answer_chain = create_stuff_documents_chain(
    chat_model,
    prompt,
)

# The retrieval chain first searches Pinecone and then asks the model
# to answer using the retrieved documents.
rag_chain = create_retrieval_chain(
    retriever,
    question_answer_chain,
)

print("MediGuide RAG chain created successfully.")

MediGuide RAG chain created successfully.


## 16. Reusable MediGuide Question Function

This helper keeps notebook testing clean and can later inspire the logic used by the Flask backend.

In [33]:
def ask_mediguide(question: str) -> str:
    """Ask MediGuide a question and return the generated answer."""

    if not question or not question.strip():
        return "Please enter a valid question."

    response = rag_chain.invoke({"input": question.strip()})
    return response["answer"]

## 17. Test MediGuide

In [35]:
question = "What is acromegaly and gigantism?"

print("Question:", question)
print("\nMediGuide:")
print(ask_mediguide(question))

Question: What is acromegaly and gigantism?

MediGuide:
Acromegaly and gigantism are disorders related to the abnormal release of a chemical from the pituitary gland in the brain, which causes increased growth in bone and soft tissue.

- **Acromegaly** occurs when this abnormality happens after bone growth has stopped. It leads to increased growth in bone and soft tissue, along with various other disturbances throughout the body. It is a relatively rare disorder, affecting approximately 50 out of every one million people. Both men and women can be affected, and symptoms develop gradually, often leading to a delayed diagnosis until middle age.

- **Gigantism** occurs when the abnormal release of the growth hormone happens before the bone growth has stopped, leading to unusual height.

If you suspect symptoms of these conditions, it is important to consult a healthcare professional for proper diagnosis and management.


In [36]:
question = "What is acne?"

print("Question:", question)
print("\nMediGuide:")
print(ask_mediguide(question))

Question: What is acne?

MediGuide:
Acne is a common skin disease characterized by pimples on the face, chest, and back. It occurs when the pores of the skin become clogged with oil, dead skin cells, and bacteria. The medical term for common acne is acne vulgaris, and it is the most common skin disease, affecting nearly 17 million people in the United States.


In [37]:
question = "What are common approaches to the treatment of acne?"

print("Question:", question)
print("\nMediGuide:")
print(ask_mediguide(question))

Question: What are common approaches to the treatment of acne?

MediGuide:
Common approaches to the treatment of acne depend on its severity—mild, moderate, or severe. For mild noninflammatory acne, treatment typically involves reducing the formation of new comedones using topical medications such as tretinoin, benzoyl peroxide, adapalene, or salicylic acid. Tretinoin is particularly effective as it increases the turnover of skin cells. If inflammation is present, topical antibiotics may be added to the regimen. Improvement is usually seen within two to four weeks.

For more severe cases, other treatments like isotretinoin may be considered, although improvement can take two or more months. It's important to note that acne tends to reappear when treatment stops, but it often improves spontaneously over time.


## 18. Inspect Retrieved Sources

During development it is useful to inspect which knowledge-base chunks were used for a question.
This makes the RAG pipeline easier to debug and evaluate.

In [ ]:
def inspect_retrieval(question: str, k: int = 3) -> None:
    """Display the top retrieved chunks for a question."""

    docs = vector_store.similarity_search(question, k=k)

    print(f"Question: {question}")
    print(f"Retrieved chunks: {len(docs)}\n")

    for i, doc in enumerate(docs, start=1):
        print(f"===== Chunk {i} =====")
        print("Source:", doc.metadata.get("source", "unknown"))
        print(doc.page_content[:800])
        print()


inspect_retrieval("What is acne?")

## 19. Next Development Steps

The notebook validates the core MediGuide RAG pipeline. The production project can then separate these responsibilities into modules such as:

```text
MediGuide/
├── app.py
├── requirements.txt
├── setup.py
├── .env
├── .gitignore
├── data/
├── research/
│   └── MediGuide_RAG_Pipeline.ipynb
├── src/
│   ├── __init__.py
│   ├── helper.py
│   └── prompt.py
├── templates/
│   └── chat.html
└── static/
```

Recommended progression:

1. Move document/embedding utilities into `src/helper.py`.
2. Move the system prompt into `src/prompt.py`.
3. Create the Flask application in `app.py`.
4. Connect the Flask chat endpoint to the RAG chain.
5. Add source display and basic error handling.
6. Evaluate retrieval quality with a small set of medical test questions.
7. Add appropriate healthcare safety messaging to the user interface.

---
### End of MediGuide RAG Development Notebook

**MediGuide — AI-Powered Analysis for Intelligent Healthcare Assistance**

Developed by **Saurabh Kumbhar (25204974)** and **Azim Hassan (25203062)**.